# 03 — Split A/B/C projeté (2035, 2050)

Diagnostic du breakeven GIS (étape 0), test de robustesse de `part_dispersee` (étape 0bis),
taux d'accès historiques et trajectoires (étapes 1-2), split A/B/C et sortie (étapes 3-4).

**Aucun fichier du pipeline GIS 2025 (`analyse_GIS_phase2/`) n'est modifié ici** — tout ce qui
suit lit ses sorties telles quelles.

In [1]:
import os
import numpy as np
import pandas as pd

BASE = os.path.dirname(os.getcwd())
GIS_OUT = os.path.join(BASE, "analyse_GIS_phase2", "output")
CSV_FINAL = os.path.join(BASE, "exctraction of data", "output", "CSV_final.csv")
MENAGES_PATH = os.path.join(os.getcwd(), "output", "menages_projetes.csv")
OUT_PATH = os.path.join(os.getcwd(), "output", "split_abc_projete.csv")

## Étape 0 — diagnostic du breakeven GIS

Sortie du breakeven GIS (classification connectable/dispersed des 697 communautés) :
`analyse_GIS_phase2/output/community_breakeven_detail_BC.csv` (une ligne par communauté) et
`share_dispersion_final_BC.csv` (agrégé par cluster). Un agrégat municipal existe déjà,
`km_to_connect_by_municipio_BC.csv` (21 lignes).

In [2]:
comm = pd.read_csv(os.path.join(GIS_OUT, "community_breakeven_detail_BC.csv"), encoding="utf-8-sig")
share_disp = pd.read_csv(os.path.join(GIS_OUT, "share_dispersion_final_BC.csv"), encoding="utf-8-sig")

print(f"community_breakeven_detail_BC.csv : {comm.shape[0]} lignes -> granularité communauté")
print(f"  municipalités distinctes (Municipio_std) : {comm['Municipio_std'].nunique()}")
print(f"share_dispersion_final_BC.csv : {share_disp.shape[0]} lignes -- 5 clusters (C1-C5), pas 6 "
      "(6 lignes de wc -l compte l'en-tête)")
share_disp

community_breakeven_detail_BC.csv : 697 lignes -> granularité communauté
  municipalités distinctes (Municipio_std) : 21
share_dispersion_final_BC.csv : 5 lignes -- 5 clusters (C1-C5), pas 6 (6 lignes de wc -l compte l'en-tête)


,Cluster,share_dispersion,extension_capex_EUR,N_connectable,N_dispersed,f_min_PV_HS_GW,f_min_HS_DIESEL_GW,f_min_BATT_HS_GWh
0,C1,0.1211,14510579.0,85,79,1.924000e-05,0.000109,0.000047
1,C2,0.1118,1923418.0,11,5,6.500000e-07,0.000012,0.000002
2,C3,0.0099,11801908.0,131,29,4.860000e-06,0.000032,0.000012
3,C4,0.0680,17188456.0,214,129,4.160000e-06,0.000072,0.000010
4,C5,0.0000,157242.0,14,0,0.000000e+00,0.000000,0.000000


### Chaque communauté porte-t-elle municipio / B / C / classe ?

`comm` porte `Municipio_std`, `dispersed` (bool), et `Panel_2024`/`Motor_2024`/`Otra_2024`
(composantes de Source B). Reconstruction : `B_comm = Panel_2024+Motor_2024+Otra_2024`,
`C_comm = HH - B_comm`.

In [3]:
comm["B_comm"] = comm["Panel_2024"] + comm["Motor_2024"] + comm["Otra_2024"]
comm["C_comm_raw"] = comm["HH"] - comm["B_comm"]

n_neg = (comm["C_comm_raw"] < 0).sum()
print(f"Communautés avec C_comm < 0 (B_comm > HH, artefact d'échelle du scaling communautaire "
      f"2012->2024) : {n_neg}/{len(comm)}, somme négative = {comm.loc[comm['C_comm_raw']<0,'C_comm_raw'].sum():.1f}")
comm["C_comm"] = comm["C_comm_raw"].clip(lower=0.0)

Communautés avec C_comm < 0 (B_comm > HH, artefact d'échelle du scaling communautaire 2012->2024) : 40/697, somme négative = -325.1


### Somme B et C sur les 697 communautés vs. 9 325 (B) et 11 389 (C) census

In [4]:
B_TARGET, C_TARGET = 9325, 11389
B_recon, C_recon = comm["B_comm"].sum(), comm["C_comm"].sum()
print(f"B reconstruit = {B_recon:.0f}   cible = {B_TARGET}   écart = {B_TARGET-B_recon:.0f} ({100*(B_TARGET-B_recon)/B_TARGET:.1f} %)")
print(f"C reconstruit = {C_recon:.0f}   cible = {C_TARGET}   écart = {C_TARGET-C_recon:.0f} ({100*(C_TARGET-C_recon)/C_TARGET:.1f} %)")

B reconstruit = 7333   cible = 9325   écart = 1992 (21.4 %)
C reconstruit = 11650   cible = 11389   écart = -261 (-2.3 %)


### Cause : où B se perd, et pourquoi C n'est pas affecté

`phase2_share_dispersion.ipynb` construit deux dataframes **indépendants** :

- `task1_community_distances_lines.csv` (source de Source C, `HH_2024_scaled`) — construit dans
  `gis_phase2_analysis.ipynb`, ne référence jamais `comunidades_electricidad_2012.csv`.
- `com2012` → `study_b` (source de Source B) — filtré par
  `com2012["Municipio_std"].isin(MUNI_TO_CLUSTER)` (cellule 16), où `Municipio_std` de `com2012`
  garde les espaces bruts (`"Bella Flor"`) alors que les clés de `MUNI_TO_CLUSTER` sont jointes
  par underscore (`"Bella_Flor"`) — seule Santa Rosa (Beni) est corrigée à la main. Le filtre ne
  matche donc **jamais** aucune municipalité multi-mots.

Les deux sont ensuite combinées (cellule 17) par **indexation, pas jointure** :
`task1_bc[BCOLS] = b_indexed.reindex(key_index).fillna(0.0).values` — ça ne supprime **aucune**
ligne de `task1_bc` (donc de Source C), ça remplit juste B à 0 là où `study_b` n'a rien. C'est
pour ça que B disparaît sans que C soit touchée : ce sont deux sources de données distinctes,
combinées par un `reindex` qui préserve toujours les lignes du côté C.

In [5]:
task1 = pd.read_csv(os.path.join(GIS_OUT, "task1_community_distances_lines.csv"), encoding="utf-8-sig")

com2012 = pd.read_csv(os.path.join(BASE, "analyse_GIS_phase2", "data", "comunidades_electricidad_2012.csv"),
                       encoding="utf-8-sig", low_memory=False)
com2012["Municipio_std"] = com2012["Municipio"].str.extract(r"- (.+)$")[0].str.strip()
com2012["Municipio_std"] = com2012["Municipio_std"].fillna(com2012["Municipio"].str.strip())
fix = (com2012["Municipio_std"] == "Santa Rosa") & (com2012["Depto"] == "Beni")
com2012.loc[fix, "Municipio_std"] = "Santa_Rosa_Beni"

muni_keys = sorted(comm["Municipio_std"].unique())
zero_b_munis = sorted(comm.groupby("Municipio_std")["B_comm"].sum().loc[lambda s: s == 0].index)
missing_keys = sorted(set(muni_keys) - set(com2012["Municipio_std"]))

print(f"Municipalités à B reconstruit = 0 (section précédente) : {len(zero_b_munis)}/21")
print(" ", zero_b_munis)
print(f"Clés absentes de comunidades_electricidad_2012.csv (mismatch underscore/espace) : {len(missing_keys)}/21")
print(" ", missing_keys)
assert set(missing_keys) == set(zero_b_munis), "l'hypothèse de cause ne couvre pas exactement les municipalités à B=0"

sub = task1[task1["Municipio_std"].isin(zero_b_munis)]
print()
print(f"task1_community_distances_lines.csv (source de C) pour ces {len(zero_b_munis)} municipalités : "
      f"{len(sub)} communautés, somme HH_2024_scaled = {sub['HH_2024_scaled'].sum():.0f} "
      "-- présent et non nul, confirmant que C n'est jamais passé par le filtre isin() cassé.")

Municipalités à B reconstruit = 0 (section précédente) : 9/21
  ['Bella_Flor', 'Nueva_Esperanza', 'Puerto_Gonzalo_Moreno', 'Puerto_Rico', 'San_Lorenzo', 'San_Pedro', 'Santa_Rosa_Pando', 'Santos_Mercado', 'Villa_Nueva']
Clés absentes de comunidades_electricidad_2012.csv (mismatch underscore/espace) : 9/21
  ['Bella_Flor', 'Nueva_Esperanza', 'Puerto_Gonzalo_Moreno', 'Puerto_Rico', 'San_Lorenzo', 'San_Pedro', 'Santa_Rosa_Pando', 'Santos_Mercado', 'Villa_Nueva']

task1_community_distances_lines.csv (source de C) pour ces 9 municipalités : 212 communautés, somme HH_2024_scaled = 2416 -- présent et non nul, confirmant que C n'est jamais passé par le filtre isin() cassé.


### Écart par municipalité vs. `CSV_final.csv`

Recoupement communauté-par-communauté (`B`, `C`) avec les totaux municipaux exacts 2024 de
`CSV_final.csv` (Source B = Motor propio+Panel solar+Otra, Source C = No tiene).

In [6]:
raw_cf = pd.read_csv(CSV_FINAL, encoding="utf-8")
mm = raw_cf[raw_cf["MUNICIPIO/TIOC"].notna() & (raw_cf["MUNICIPIO/TIOC"].astype(str).str.strip() != "")].copy()
assert len(mm) == 21

def std_muni_name(depto, raw_name):
    raw_name = str(raw_name).strip()
    if raw_name == "Santa Rosa":
        return "Santa_Rosa_Beni" if depto == "Beni" else "Santa_Rosa_Pando"
    return raw_name.replace(" ", "_")

mm["Municipio_std"] = [std_muni_name(d, r) for d, r in zip(mm["DEPARTAMENTO"], mm["MUNICIPIO/TIOC"])]
HH_BLOCK = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD"
mm["B_census"] = (mm[f"{HH_BLOCK} | 2024 | Motor propio"] + mm[f"{HH_BLOCK} | 2024 | Panel solar"]
                   + mm[f"{HH_BLOCK} | 2024 | Otra"])
mm["C_census"] = mm[f"{HH_BLOCK} | 2024 | No tiene"]

by_muni = comm.groupby("Municipio_std").agg(B=("B_comm", "sum"), C=("C_comm", "sum")).reset_index()
cmp = by_muni.merge(mm[["Municipio_std", "B_census", "C_census"]], on="Municipio_std", how="outer", indicator=True)
assert (cmp["_merge"] == "both").all()
cmp["B_gap"], cmp["C_gap"] = cmp["B_census"] - cmp["B"], cmp["C_census"] - cmp["C"]
cmp.sort_values("B_gap", ascending=False)[["Municipio_std", "B", "B_census", "B_gap", "C", "C_census", "C_gap"]]

,Municipio_std,B,B_census,B_gap,C,C_census,C_gap
11,Puerto_Rico,0.000000,320.0,320.000000,325.000000,325.0,0.000000
0,Bella_Flor,0.000000,315.0,315.000000,351.000000,351.0,0.000000
8,Nueva_Esperanza,0.000000,267.0,267.000000,151.000000,151.0,0.000000
14,San_Lorenzo,0.000000,260.0,260.000000,302.000000,302.0,0.000000
15,San_Pedro,0.000000,241.0,241.000000,369.000000,369.0,0.000000
10,Puerto_Gonzalo_Moreno,0.000000,232.0,232.000000,348.000000,348.0,0.000000
20,Villa_Nueva,0.000000,221.0,221.000000,213.000000,213.0,0.000000
18,Santos_Mercado,0.000000,205.0,205.000000,208.000000,208.0,0.000000
17,Santa_Rosa_Pando,0.000000,143.0,143.000000,149.000000,149.0,0.000000
7,Ixiamas,741.910051,793.0,51.089949,1086.604712,1043.0,-43.604712


### Conclusion étape 0

Granularité communauté disponible (697 lignes), avec municipio, classe et, en principe, B/C.
Mais la reconstruction communauté-par-communauté de Source B est cassée par un bug de jointure
amont (`isin()` underscore/espace) : 9/21 municipalités à B=0 alors que leur total municipal réel
est non nul. Source C n'est pas affectée (vient d'une source de données indépendante). Sur les
12 municipalités restantes, l'appariement communauté-par-communauté (nom 2012 vs GIS) perd
encore une partie de B (jusqu'à -222 ménages, Riberalta). Fichiers `analyse_GIS_phase2/`
non modifiés — test de robustesse ci-dessous avant de décider comment poursuivre.

## Étape 0bis — test de robustesse de `part_dispersee`

Sur les 12 municipalités où Source B est correctement rattachée à une communauté (B ≠ 0), on
compare deux façons de calculer `part_dispersee(m) = (B+C en communautés dispersed)/(B+C total)` :

- **réelle** : `HH_dispersed(m) / HH(m)` avec les vraies valeurs de B par communauté (`HH` du
  fichier breakeven, B inclus)
- **prorata** : on ignore B au niveau communauté et on répartit le B municipal au prorata de C
  dans chaque communauté avant d'appliquer la même formule — algébriquement, le facteur
  `(1 + B_census(m)/C(m))` étant constant par municipalité, ça revient exactement à
  `C_disp_comm(m) / C_comm(m)` (le B ajouté uniformément ne change pas le ratio)

Seuil de décision fixé à l'avance : écart moyen absolu < 5 points -> méthode prorata sur les 21
municipalités. >= 5 points -> arrêt, pas de méthode de repli choisie ici.

In [7]:
zero_b_munis_set = set(zero_b_munis)
munis_12 = [m for m in muni_keys if m not in zero_b_munis_set]

rows_0bis = []
for muni in munis_12:
    sub = comm[comm["Municipio_std"] == muni]
    HH_tot, HH_disp = sub["HH"].sum(), sub.loc[sub["dispersed"], "HH"].sum()
    C_tot, C_disp = sub["C_comm"].sum(), sub.loc[sub["dispersed"], "C_comm"].sum()
    rows_0bis.append(dict(
        municipio=muni, part_reelle=HH_disp / HH_tot, part_prorata=C_disp / C_tot,
    ))
robustesse = pd.DataFrame(rows_0bis)
robustesse["ecart_pp"] = 100 * (robustesse["part_prorata"] - robustesse["part_reelle"])
robustesse = robustesse.sort_values("ecart_pp", key=abs, ascending=False)

ecart_moyen_abs = robustesse["ecart_pp"].abs().mean()
print(f"12 municipalités testées, écart moyen absolu = {ecart_moyen_abs:.2f} points de pourcentage")
robustesse.round(4)

12 municipalités testées, écart moyen absolu = 3.21 points de pourcentage


,municipio,part_reelle,part_prorata,ecart_pp
2,Exaltación,0.2346,0.3602,12.5621
5,Ingavi,0.1932,0.3036,11.0424
0,Bolpebra,0.1387,0.2086,6.9834
10,Santa_Rosa_Beni,0.2195,0.2467,2.7260
11,Sena,0.3083,0.3351,2.6719
4,Guayaramerín,0.0914,0.0748,-1.6598
9,Riberalta,0.0569,0.0624,0.5475
8,Reyes,0.3120,0.3134,0.1366
3,Filadelfia,0.0113,0.0125,0.1177
6,Ixiamas,0.2894,0.2886,-0.0731


In [8]:
ecart_moyen_signe = robustesse["ecart_pp"].mean()
n_pos = (robustesse["ecart_pp"] > 0).sum()
print(f"écart moyen SIGNÉ = {ecart_moyen_signe:+.2f} points ({n_pos}/{len(robustesse)} municipalités positives)")

écart moyen SIGNÉ = +2.92 points (8/12 municipalités positives)


Biais directionnel (écart moyen signé +2.92 points, 8/12 municipalités positives) : la méthode prorata surestime systématiquement `part_dispersee`, donc surestime B et sous-estime A — cohérent avec le fait que les ménages Source B, déjà équipés, tendent à vivre dans des communautés moins isolées que Source C. Ce biais s'ajoute, dans le même sens, à celui du breakeven recalculé sur des communautés plus grosses (plus susceptibles d'être classées connectable) — non corrigé ici, documenté seulement.

In [9]:
SEUIL_PP = 5.0
assert ecart_moyen_abs < SEUIL_PP, (
    f"écart moyen absolu {ecart_moyen_abs:.2f} pp >= seuil {SEUIL_PP} pp -- ARRÊT, "
    "pas de méthode de repli choisie ici."
)
print(f"écart moyen absolu {ecart_moyen_abs:.2f} pp < seuil {SEUIL_PP} pp -> méthode PRORATA retenue, "
      "appliquée aux 21 municipalités.")

écart moyen absolu 3.21 pp < seuil 5.0 pp -> méthode PRORATA retenue, appliquée aux 21 municipalités.


**Méthode retenue : prorata.** Hypothèse posée : les ménages Source B sont répartis dans
l'espace intra-municipal comme les ménages Source C (distribution intra-municipale supposée
stationnaire, mémoire §4.8.1). `part_dispersee(m) = C_disp_comm(m) / C_comm(m)`, calculée pour
les 21 municipalités (pour les 9 à B=0, c'est déjà exactement ce que fait la donnée : `C_comm =
HH`, donc identique à la part réelle).

In [10]:
pd_rows = []
for muni, sub in comm.groupby("Municipio_std"):
    C_tot = sub["C_comm"].sum()
    C_disp = sub.loc[sub["dispersed"], "C_comm"].sum()
    pd_rows.append(dict(Municipio_std=muni, part_dispersee=(C_disp / C_tot if C_tot > 0 else 0.0)))
part_disp = pd.DataFrame(pd_rows)

assert len(part_disp) == 21
assert part_disp["part_dispersee"].between(0, 1).all()
part_disp.sort_values("part_dispersee")

,Municipio_std,part_dispersee
2,Cobija,0.000000
9,Porvenir,0.000000
10,Puerto_Gonzalo_Moreno,0.000000
4,Filadelfia,0.012506
13,Riberalta,0.062391
15,San_Pedro,0.063981
5,Guayaramerín,0.074792
17,Santa_Rosa_Pando,0.118182
0,Bella_Flor,0.147186
1,Bolpebra,0.208572


## Étape 1 — taux d'accès historiques

Source : `CSV_final.csv`, filtre `MUNICIPIO/TIOC` non vide (élimine les 8 sous-totaux). Taux
d'accès = part des viviendas avec électricité, publique ou privée :

- 2001 : `Tiene eléctrica / Electricidad total` (le recensement 2001 ne distingue pas la source)
- 2012, 2024 : `(Total - No tiene) / Total`

In [11]:
m = mm
assert len(m) == 21

acc = pd.DataFrame({
    "departamento": m["DEPARTAMENTO"], "provincia": m["PROVINCIA"], "municipio": m["MUNICIPIO/TIOC"],
    "Municipio_std": m["Municipio_std"],
})
acc["acc_2001"] = m[f"{HH_BLOCK} | 2001 | Tiene eléctrica"] / m[f"{HH_BLOCK} | 2001 | Electricidad total"]
acc["acc_2012"] = (m[f"{HH_BLOCK} | 2012 | Total"] - m[f"{HH_BLOCK} | 2012 | No tiene"]) / m[f"{HH_BLOCK} | 2012 | Total"]
acc["acc_2024"] = (m[f"{HH_BLOCK} | 2024 | Total"] - m[f"{HH_BLOCK} | 2024 | No tiene"]) / m[f"{HH_BLOCK} | 2024 | Total"]
acc = acc.reset_index(drop=True)

assert acc[["acc_2001", "acc_2012", "acc_2024"]].apply(lambda s: s.between(0, 1)).all().all()
acc[["departamento", "municipio", "acc_2001", "acc_2012", "acc_2024"]].round(4)

,departamento,municipio,acc_2001,acc_2012,acc_2024
0,La Paz,Ixiamas,0.2431,0.5656,0.6845
1,Beni,Riberalta,0.4626,0.8692,0.8934
2,Beni,Guayaramerín,0.6714,0.8962,0.9242
3,Beni,Reyes,0.4556,0.6779,0.7919
4,Beni,Santa Rosa,0.3937,0.8426,0.8523
5,Beni,Exaltación,0.1688,0.7111,0.8124
6,Pando,Cobija,0.7567,0.9173,0.9746
7,Pando,Porvenir,0.4296,0.7644,0.9045
8,Pando,Bolpebra,0.0856,0.4422,0.6845
9,Pando,Bella Flor,0.0604,0.5561,0.7158


## Étape 2 — trajectoires du taux d'accès

**Accès 2035** (volontariste) : fermeture linéaire du gap 2024→2035, normalisée pour atteindre
exactement 1 à 2035 (`phi_2035_traj(m,t) = min((t-2024)/11, 1)`), puis maintien à 1 jusqu'en 2050
(la même formule, une fois `phi=1`, reste à 1 pour t > 2035).

**Accès 2050** (tendancielle) : taux de fermeture annuel calé sur 2012→2024 uniquement,
`r = 1 - ((1-acc_2024)/(1-acc_2012))^(1/12)`. Cas `r <= 0` : forcé à `r_final=0`, signalé
nommément (n'avance que par la contrainte d'atteinte de 100 % en 2050, pas par tendance).

**Correction du ré-échelonnage** (le taux constant `r_final` avançait au rythme naturel puis
un forçage `acc(2050)=1.0` sans lien avec `r_final` créait une discontinuité de rythme après
2035) : la forme à taux constant est conservée mais normalisée pour atteindre exactement 1 en
2050 par construction, pas par un forçage littéral :

    phi(m,t) = (1 - (1-r_final)^(t-2024)) / (1 - (1-r_final)^(2050-2024))
    acc(m,t) = acc_2024 + phi(m,t) * (1 - acc_2024)

Cas `r_final = 0` (dénominateur nul) : fermeture linéaire du gap sur 2024-2050,
`phi(m,t) = (t-2024)/26`.

In [12]:
acc["r"] = 1 - ((1 - acc["acc_2024"]) / (1 - acc["acc_2012"])) ** (1 / 12)
r_non_positif = acc.loc[acc["r"] <= 0, ["municipio", "r"]]
print(f"Municipalité(s) avec r <= 0, forcées à r=0 : {len(r_non_positif)}")
print(r_non_positif.to_string(index=False))

sena = acc.loc[acc["municipio"] == "Sena"].iloc[0]
print(f"\nSena -- acc_2001={sena['acc_2001']:.3f} acc_2012={sena['acc_2012']:.3f} "
      f"acc_2024={sena['acc_2024']:.3f} r={sena['r']:.4f} "
      "(recul 2001->2012, mais calé sur 2012->2024 uniquement -> r positif, normal)")

acc["r_final"] = np.where(acc["r"] <= 0, 0.0, acc["r"])
acc[["municipio", "acc_2012", "acc_2024", "r", "r_final"]].sort_values("r")

Municipalité(s) avec r <= 0, forcées à r=0 : 1
municipio         r
   Ingavi -0.002661

Sena -- acc_2001=0.183 acc_2012=0.140 acc_2024=0.627 r=0.0672 (recul 2001->2012, mais calé sur 2012->2024 uniquement -> r positif, normal)


,municipio,acc_2012,acc_2024,r,r_final
17,Ingavi,0.610465,0.597843,-0.002661,0.000000
4,Santa Rosa,0.842611,0.852269,0.005263,0.005263
11,San Pedro,0.284779,0.397059,0.014130,0.014130
1,Riberalta,0.869206,0.893412,0.016910,0.016910
2,Guayaramerín,0.896232,0.924249,0.025885,0.025885
0,Ixiamas,0.565639,0.684513,0.026295,0.026295
20,Santos Mercado,0.500000,0.653910,0.030194,0.030194
19,Villa Nueva,0.563667,0.699153,0.030508,0.030508
5,Exaltación,0.711087,0.812371,0.035332,0.035332
3,Reyes,0.677920,0.791923,0.035753,0.035753


In [13]:
def phi_2050(r_final, t):
    if r_final == 0.0:
        return (t - 2024) / 26
    return (1 - (1 - r_final) ** (t - 2024)) / (1 - (1 - r_final) ** 26)


acc["phi_2035"] = [phi_2050(r, 2035) for r in acc["r_final"]]
print("phi_2035 : fraction du gap 2024->2050 comblee des 2035, trajectoire tendancielle "
      "(diagnostic, remplace l'ancien effort_suppl_2050_pp qui n'a plus de sens une fois "
      "la trajectoire normalisee) :")
acc[["municipio", "r_final", "phi_2035"]].sort_values("phi_2035", ascending=False).round(4)

phi_2035 : fraction du gap 2024->2050 comblee des 2035, trajectoire tendancielle (diagnostic, remplace l'ancien effort_suppl_2050_pp qui n'a plus de sens une fois la trajectoire normalisee) :


,municipio,r_final,phi_2035
6,Cobija,0.0938,0.7169
14,San Lorenzo,0.0774,0.6703
7,Porvenir,0.0725,0.6557
15,Sena,0.0672,0.6395
13,Puerto Gonzalo Moreno,0.0575,0.6095
10,Puerto Rico,0.0571,0.6080
16,Santa Rosa,0.0540,0.5982
12,Filadelfia,0.0526,0.5939
8,Bolpebra,0.0464,0.5739
18,Nueva Esperanza,0.0421,0.5598


In [14]:
TRAJ_YEARS = [2024, 2035, 2050]
acc_long_rows = []
for _, row in acc.iterrows():
    for y in TRAJ_YEARS:
        frac35 = min((y - 2024) / (2035 - 2024), 1.0)
        a35 = row["acc_2024"] + frac35 * (1 - row["acc_2024"])
        acc_long_rows.append(dict(Municipio_std=row["Municipio_std"], annee=y, trajectoire="acces_2035", acc=a35))
    for y in TRAJ_YEARS:
        a50 = row["acc_2024"] + phi_2050(row["r_final"], y) * (1 - row["acc_2024"])
        acc_long_rows.append(dict(Municipio_std=row["Municipio_std"], annee=y, trajectoire="acces_2050", acc=a50))
acc_long = pd.DataFrame(acc_long_rows)

assert acc_long["acc"].between(-1e-12, 1 + 1e-12).all()
acc_long["acc"] = acc_long["acc"].clip(0, 1)

# Assertions bloquantes de la correction (a 1e-12, jamais un forcage litteral)
piv_check = acc_long.merge(acc[["Municipio_std", "acc_2024", "r_final"]], on="Municipio_std").pivot_table(
    index=["Municipio_std", "acc_2024", "r_final"], columns=["trajectoire", "annee"], values="acc").reset_index()

assert np.allclose(piv_check[("acces_2035", 2024)], piv_check["acc_2024"], atol=1e-12)
assert np.allclose(piv_check[("acces_2050", 2024)], piv_check["acc_2024"], atol=1e-12)
assert np.allclose(piv_check[("acces_2035", 2050)], 1.0, atol=1e-12)
assert np.allclose(piv_check[("acces_2050", 2050)], 1.0, atol=1e-12)

r_pos = piv_check["r_final"] > 0
assert (piv_check.loc[r_pos, ("acces_2050", 2035)] > piv_check.loc[r_pos, "acc_2024"] + 1e-12).all()

for traj in ["acces_2035", "acces_2050"]:
    piv = acc_long[acc_long["trajectoire"] == traj].merge(
        acc[["Municipio_std", "departamento"]], on="Municipio_std"
    ).pivot(index=["departamento", "Municipio_std"], columns="annee", values="acc")
    assert (piv[2035] >= piv[2024] - 1e-9).all(), "non-régression violée 2024->2035"
    assert (piv[2050] >= piv[2035] - 1e-9).all(), "non-régression violée 2035->2050"
print("Assertions de la correction (acc(2024)==acc_2024, acc(2050)==1.0 a 1e-12, "
      "acc(2035)>acc_2024 pour r_final>0, non-regression) : OK pour les deux trajectoires.")

Assertions de la correction (acc(2024)==acc_2024, acc(2050)==1.0 a 1e-12, acc(2035)>acc_2024 pour r_final>0, non-regression) : OK pour les deux trajectoires.


## Étape 3 — split A/B/C

### Deuxième correction : `part_dispersee` ne s'applique pas au stock B de 2024

En 2024, dans le bassin non raccordé, avoir déjà l'électricité **fait** d'un ménage un ménage
Source B — `part_dispersee` ne décrit pas qui est B aujourd'hui, elle décrit l'état final, une
fois toutes les lignes économiquement rentables construites. Les ménages B situés en communauté
connectable basculent vers A **progressivement**, au rythme de construction des lignes — pas en
2024, pas jamais pour les ménages en communauté dispersée.

```
g(m,t)      = hh(m,t) / hh(m,2024)
pool(m,t)   = (B0(m)+C0(m)) * g(m,t)
acc_pool(m,t) : taux d'accès du bassin (inchangé, acc_pool(m,2024) = B0/(B0+C0) exact)

f(m,t) = (acc_pool(m,t) - acc_pool(m,2024)) / (1 - acc_pool(m,2024))     [borné [0,1]]

B0_disp(m) = part_dispersee(m) * B0(m)          -- reste B pour toujours
B0_conn(m) = (1 - part_dispersee(m)) * B0(m)     -- bascule vers A au rythme f(m,t)
nouv(m,t)  = acc_pool(m,t)*pool(m,t) - B0(m)*g(m,t)     -- nouvellement électrifiés, borné à 0

hh_B(m,t) = [B0_disp(m) + B0_conn(m)*(1-f(m,t))] * g(m,t)  +  part_dispersee(m) * nouv(m,t)
hh_C(m,t) = (1 - acc_pool(m,t)) * pool(m,t)
hh_A(m,t) = hh(m,t) - hh_B(m,t) - hh_C(m,t)
```

`hh(m,t)` vient de `menages_projetes.csv`, jamais recalculé. `part_dispersee` reste gelée
(méthode prorata, étape 0bis).</cell id="ce3152c6">


In [15]:
mm["A0"] = mm[f"{HH_BLOCK} | 2024 | Servicio público de energía eléctrica"]
mm["B0"] = mm["B_census"]
mm["C0"] = mm["C_census"]

base = acc[["departamento", "provincia", "municipio", "Municipio_std"]].merge(
    part_disp, on="Municipio_std", how="left"
).merge(mm[["Municipio_std", "A0", "B0", "C0"]], on="Municipio_std", how="left")
assert base[["part_dispersee", "A0", "B0", "C0"]].notna().all().all()
assert len(base) == 21

hh_proj = pd.read_csv(MENAGES_PATH)
hh_long = hh_proj.melt(id_vars=["departamento", "provincia", "municipio", "cluster"],
                        value_vars=["hh_2024", "hh_2035", "hh_2050"], var_name="col", value_name="hh_total")
hh_long["annee"] = hh_long["col"].str.replace("hh_", "").astype(int)
hh_long = hh_long.drop(columns="col")
hh_2024_map = hh_proj.set_index(["departamento", "provincia", "municipio"])["hh_2024"]

out_rows = []
for traj in ["acces_2035", "acces_2050"]:
    sub = acc_long[acc_long["trajectoire"] == traj]
    tmp = base.merge(sub, on="Municipio_std", how="left")
    tmp = tmp.merge(hh_long, on=["departamento", "provincia", "municipio", "annee"], how="left", indicator="_mrg")
    assert (tmp["_mrg"] == "both").all(), tmp.loc[tmp["_mrg"] != "both", ["departamento", "municipio", "annee"]]
    out_rows.append(tmp.drop(columns="_mrg"))
out = pd.concat(out_rows, ignore_index=True)

out["hh_2024"] = out.set_index(["departamento", "provincia", "municipio"]).index.map(hh_2024_map)
out["g"] = out["hh_total"] / out["hh_2024"]
out["A0_grown"] = out["A0"] * out["g"]
out["pool"] = (out["B0"] + out["C0"]) * out["g"]
out["acc_pool"] = (out["acc"] * out["hh_total"] - out["A0_grown"]) / out["pool"]

acc_pool_2024 = out[out["annee"] == 2024].drop_duplicates(subset=["departamento", "provincia", "municipio"]) \
    .set_index(["departamento", "provincia", "municipio"])["acc_pool"]
out["acc_pool_2024"] = out.set_index(["departamento", "provincia", "municipio"]).index.map(acc_pool_2024)

denom = 1 - out["acc_pool_2024"]
out["f"] = np.where(denom.abs() < 1e-12, 1.0, (out["acc_pool"] - out["acc_pool_2024"]) / denom)
out["f"] = out["f"].clip(0, 1)

out["B0_disp"] = out["part_dispersee"] * out["B0"]
out["B0_conn"] = (1 - out["part_dispersee"]) * out["B0"]
out["nouv_raw"] = out["acc_pool"] * out["pool"] - out["B0"] * out["g"]
n_neg_nouv = (out["nouv_raw"] < -1e-6).sum()
print(f"nouv négatif avant borne à 0 : {n_neg_nouv} ligne(s)"
      + ("" if n_neg_nouv == 0 else " -- SIGNALÉ, à examiner"))
out["nouv"] = out["nouv_raw"].clip(lower=0.0)

out["hh_B"] = (out["B0_disp"] + out["B0_conn"] * (1 - out["f"])) * out["g"] + out["part_dispersee"] * out["nouv"]
out["hh_C"] = (1 - out["acc_pool"]) * out["pool"]
out["hh_A"] = out["hh_total"] - out["hh_B"] - out["hh_C"]

out = out[["departamento", "provincia", "municipio", "cluster", "trajectoire", "annee",
           "hh_total", "acc", "part_dispersee", "acc_pool", "f", "A0", "B0", "C0",
           "hh_A", "hh_B", "hh_C"]]
print(f"shape = {out.shape} (attendu (126, 17))")
out.head()

nouv négatif avant borne à 0 : 0 ligne(s)
shape = (126, 17) (attendu (126, 17))


,departamento,provincia,municipio,cluster,trajectoire,annee,hh_total,acc,part_dispersee,acc_pool,f,A0,B0,C0,hh_A,hh_B,hh_C
0,La Paz,Abel Iturralde,Ixiamas,C1,acces_2035,2024,3306.000000,0.684513,0.288625,0.431917,0.0,1470.0,793.0,1043.0,1470.000000,793.000000,1.043000e+03
1,La Paz,Abel Iturralde,Ixiamas,C1,acces_2035,2035,4059.330684,1.000000,0.288625,1.000000,1.0,1470.0,793.0,1043.0,3408.663423,650.667261,-5.005696e-13
2,La Paz,Abel Iturralde,Ixiamas,C1,acces_2035,2050,4811.556447,1.000000,0.288625,1.000000,1.0,1470.0,793.0,1043.0,4040.315448,771.241000,0.000000e+00
3,Beni,Vaca Diez,Riberalta,C3,acces_2035,2024,27442.000000,0.893412,0.062391,0.323073,0.0,23121.0,1396.0,2925.0,23121.000000,1396.000000,2.925000e+03
4,Beni,Vaca Diez,Riberalta,C3,acces_2035,2035,33049.840302,1.000000,0.062391,1.000000,1.0,23121.0,1396.0,2925.0,32725.159112,324.681191,1.155522e-12


### Vérification : `acc_pool(m, 2024)` reproduit-il la part de B dans B+C au recensement ?

À 2024, `acc(m,2024)*hh(m,2024) - A0(m) = B_census(m)` exactement (par construction de
`acc_2024`), et `pool(m,2024) = hh(m,2024) - A0(m) = B_census(m) + C_census(m)`. Donc
`acc_pool(m,2024)` doit égaler `B_census(m) / (B_census(m) + C_census(m))` pour les 21
municipalités.

In [16]:
check24 = out[(out["annee"] == 2024) & (out["trajectoire"] == "acces_2035")].merge(
    acc[["departamento", "provincia", "municipio", "Municipio_std"]], on=["departamento", "provincia", "municipio"]
).merge(mm[["Municipio_std", "B_census", "C_census"]], on="Municipio_std")

check24["b_share_census"] = check24["B_census"] / (check24["B_census"] + check24["C_census"])
check24["diff"] = check24["acc_pool"] - check24["b_share_census"]

assert len(check24) == 21
assert check24["diff"].abs().max() < 1e-9, check24.loc[check24["diff"].abs() >= 1e-9]
print(f"max |acc_pool(2024) - B_census/(B_census+C_census)| = {check24['diff'].abs().max():.2e} -- OK")
check24[["municipio", "acc_pool", "b_share_census"]].round(4)

max |acc_pool(2024) - B_census/(B_census+C_census)| = 0.00e+00 -- OK


,municipio,acc_pool,b_share_census
0,Ixiamas,0.4319,0.4319
1,Riberalta,0.3231,0.3231
2,Guayaramerín,0.4392,0.4392
3,Reyes,0.3318,0.3318
4,Santa Rosa,0.6037,0.6037
5,Exaltación,0.7653,0.7653
6,Cobija,0.3857,0.3857
7,Porvenir,0.5047,0.5047
8,Bolpebra,0.6084,0.6084
9,Bella Flor,0.4730,0.4730


### Assertions bloquantes — reconstruction exacte de 2024

Avec la formule corrigée, `f(m,2024)=0` et `nouv(m,2024)=0` par construction (démontré dans le
markdown ci-dessus), donc `hh_B(m,2024)=B0(m)`, `hh_C(m,2024)=C0(m)`, `hh_A(m,2024)=A0(m)`
**exactement**, pour chacune des 21 municipalités — plus une approximation. C'est l'assertion qui
valide la correction.

In [17]:
d24 = out[(out["annee"] == 2024) & (out["trajectoire"] == "acces_2035")]
assert len(d24) == 21

assert (abs(d24["hh_B"] - d24["B0"]) <= 1).all(), d24.loc[abs(d24["hh_B"] - d24["B0"]) > 1]
assert (abs(d24["hh_A"] - d24["A0"]) <= 1).all(), d24.loc[abs(d24["hh_A"] - d24["A0"]) > 1]
assert (abs(d24["hh_C"] - d24["C0"]) <= 1).all(), d24.loc[abs(d24["hh_C"] - d24["C0"]) > 1]

A24, B24, C24 = d24["hh_A"].sum(), d24["hh_B"].sum(), d24["hh_C"].sum()
print(f"2024 reconstruit : A={A24:.0f} B={B24:.0f} C={C24:.0f} total={A24+B24+C24:.0f}")
print(f"2024 census      : A=63495 B=9325 C=11389 total=84209")
print("Assertions 1-3 (hh_B=B0, hh_A=A0, hh_C=C0 à ±1) : OK pour les 21 municipalités.")

2024 reconstruit : A=63495 B=9325 C=11389 total=84209
2024 census      : A=63495 B=9325 C=11389 total=84209
Assertions 1-3 (hh_B=B0, hh_A=A0, hh_C=C0 à ±1) : OK pour les 21 municipalités.


### Assertion 4 — le programme de lignes atteint 100 % à l'année cible

`f(m, année cible)` doit valoir 1 pour les 21 municipalités de chaque trajectoire (2035 pour
`acces_2035`, 2050 pour `acces_2050`).

In [18]:
for traj, target_year in [("acces_2035", 2035), ("acces_2050", 2050)]:
    sub = out[(out["trajectoire"] == traj) & (out["annee"] == target_year)]
    assert (abs(sub["f"] - 1.0) <= 1e-6).all(), sub.loc[abs(sub["f"] - 1.0) > 1e-6]
    print(f"{traj}, {target_year} : f min={sub['f'].min():.6f} max={sub['f'].max():.6f} -- OK")

acces_2035, 2035 : f min=1.000000 max=1.000000 -- OK
acces_2050, 2050 : f min=1.000000 max=1.000000 -- OK


## Étape 4 — sortie et assertions bloquantes

In [19]:
OUT_COLS = ["departamento", "provincia", "municipio", "cluster", "trajectoire", "annee",
            "hh_total", "acc", "part_dispersee", "hh_A", "hh_B", "hh_C"]
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
out[OUT_COLS].to_csv(OUT_PATH, index=False)
print(f"écrit : {OUT_PATH}  ({len(out)} lignes, {len(OUT_COLS)} colonnes)")

écrit : c:\Valen\Tfe\bolivia-energy-data\projections\output\split_abc_projete.csv  (126 lignes, 12 colonnes)


In [20]:
assert len(base) == 21
assert (abs(out["hh_A"] + out["hh_B"] + out["hh_C"] - out["hh_total"]) <= 1).all()

for traj, target_year in [("acces_2035", 2035), ("acces_2050", 2050)]:
    sub = out[(out["trajectoire"] == traj) & (out["annee"] >= target_year)]
    assert (sub["hh_C"].abs() <= 1e-6).all(), sub

assert out["acc"].between(0, 1).all()
assert out["part_dispersee"].between(0, 1).all()
assert out["acc_pool"].between(-1e-9, 1 + 1e-9).all()
assert out["f"].between(-1e-9, 1 + 1e-9).all()
assert (out["hh_A"] >= -1e-6).all() and (out["hh_B"] >= -1e-6).all() and (out["hh_C"] >= -1e-6).all()

for traj in ["acces_2035", "acces_2050"]:
    piv = out[out["trajectoire"] == traj].pivot(index=["departamento", "municipio"], columns="annee", values="acc")
    assert (piv[2035] >= piv[2024] - 1e-9).all()
    assert (piv[2050] >= piv[2035] - 1e-9).all()

assert out.isna().sum().sum() == 0
assert len(out) == 21 * 3 * 2 == 126

print("TOUTES LES ASSERTIONS BLOQUANTES SONT PASSÉES.")

TOUTES LES ASSERTIONS BLOQUANTES SONT PASSÉES.


### Contrôles de cohérence

**a.** Les deux trajectoires doivent donner exactement le même split en 2050 (même
`part_dispersee`, `acc=1`, mêmes `hh`), ligne par ligne par municipalité.
**b.** `hh_B` peut désormais reculer : les ménages B en communauté connectable basculent vers A
au rythme de `f`. Ce n'est pas une erreur — liste des municipalités concernées, sans assertion.

In [21]:
c2050_35 = out[(out["annee"] == 2050) & (out["trajectoire"] == "acces_2035")].set_index(
    ["departamento", "municipio"])[["hh_A", "hh_B", "hh_C"]]
c2050_50 = out[(out["annee"] == 2050) & (out["trajectoire"] == "acces_2050")].set_index(
    ["departamento", "municipio"])[["hh_A", "hh_B", "hh_C"]]
diff_2050 = (c2050_35 - c2050_50).abs().max().max()
assert diff_2050 < 1e-6, (c2050_35 - c2050_50).abs()
print(f"a. écart max entre trajectoires en 2050 = {diff_2050:.2e} -- identiques")

print("\nb. municipalités où hh_B recule, par trajectoire et période :")
for traj in ["acces_2035", "acces_2050"]:
    piv = out[out["trajectoire"] == traj].pivot(index=["departamento", "municipio"], columns="annee", values="hh_B")
    for y0, y1 in [(2024, 2035), (2035, 2050)]:
        dec = (piv[y1] - piv[y0]).sort_values()
        dec = dec[dec < -1e-6]
        print(f"  {traj} {y0}->{y1} : {len(dec)} municipalité(s)")
        if len(dec):
            print(dec.round(0).to_string())

a. écart max entre trajectoires en 2050 = 0.00e+00 -- identiques

b. municipalités où hh_B recule, par trajectoire et période :
  acces_2035 2024->2035 : 18 municipalité(s)
departamento  municipio            
Beni          Riberalta               -1071.0
Pando         Filadelfia               -589.0
Beni          Guayaramerín             -521.0
              Exaltación               -366.0
              Santa Rosa               -308.0
Pando         Cobija                   -248.0
              Puerto Gonzalo Moreno    -232.0
              Bolpebra                 -225.0
              Porvenir                 -217.0
              Bella Flor               -207.0
              San Pedro                -191.0
La Paz        Ixiamas                  -142.0
Pando         Nueva Esperanza          -114.0
              Santa Rosa               -100.0
              San Lorenzo               -99.0
              Villa Nueva               -67.0
              Ingavi                    -65.0
         

### Totaux agrégés par cluster et pour la région

In [22]:
agg_cluster = out.groupby(["cluster", "trajectoire", "annee"])[["hh_total", "hh_A", "hh_B", "hh_C"]].sum().round(0)
agg_cluster

hh_total     hh_A    hh_B    hh_C
cluster trajectoire annee                                   
C1      acces_2035  2024    10933.0   5843.0  2656.0  2434.0
                    2035    12942.0  11098.0  1845.0    -0.0
                    2050    14873.0  12722.0  2151.0    -0.0
        acces_2050  2024    10933.0   5843.0  2656.0  2434.0
                    2035    12942.0   8984.0  2538.0  1420.0
                    2050    14873.0  12722.0  2151.0    -0.0
C2      acces_2035  2024      802.0    156.0   393.0   253.0
                    2035      998.0    830.0   168.0     0.0
                    2050     1204.0   1002.0   202.0     0.0
        acces_2050  2024      802.0    156.0   393.0   253.0
                    2035      998.0    559.0   305.0   134.0
                    2050     1204.0   1002.0   202.0     0.0
C3      acces_2035  2024    40298.0  33926.0  2274.0  4098.0
                    2035    47958.0  47508.0   450.0     0.0
                    2050    55641.0  55122.0   519.0    -0.0
        acces_2050  2024    40298.0  33926.0  2274.0  4098.0
                    2035    47958.0  43904.0  1575.0  2478.0
                    2050    55641.0  55122.0   519.0    -0.0
C4      acces_2035  2024    16612.0   8649.0  3754.0  4209.0
                    2035    20801.0  18651.0  2150.0     0.0
                    2050    25487.0  22878.0  2610.0    -0.0
        acces_2050  2024    16612.0   8649.0  3754.0  4209.0
                    2035    20801.0  15386.0  3216.0  2200.0
                    2050    25487.0  22878.0  2610.0    -0.0
C5      acces_2035  2024    15564.0  14921.0   248.0   395.0
                    2035    19084.0  19084.0     0.0     0.0
                    2050    22994.0  22994.0     0.0     0.0
        acces_2050  2024    15564.0  14921.0   248.0   395.0
                    2035    19084.0  18861.0    86.0   137.0
                    2050    22994.0  22994.0     0.0     0.0

In [23]:
agg_region = out.groupby(["trajectoire", "annee"])[["hh_total", "hh_A", "hh_B", "hh_C"]].sum().round(0)
agg_region

hh_total      hh_A    hh_B     hh_C
trajectoire annee                                     
acces_2035  2024    84209.0   63495.0  9325.0  11389.0
            2035   101783.0   97171.0  4612.0      0.0
            2050   120199.0  114718.0  5481.0     -0.0
acces_2050  2024    84209.0   63495.0  9325.0  11389.0
            2035   101783.0   87694.0  7720.0   6369.0
            2050   120199.0  114718.0  5481.0     -0.0

In [24]:
MENAGES_TARGET = {2024: 84209, 2035: 101783, 2050: 120199}
cluster_sum = agg_cluster.groupby(["trajectoire", "annee"])["hh_total"].sum()
for (traj, y), region_total in agg_region["hh_total"].items():
    assert abs(cluster_sum[(traj, y)] - region_total) < 1e-6
    assert abs(region_total - MENAGES_TARGET[y]) < 1e-6, (traj, y, region_total, MENAGES_TARGET[y])
print("somme clusters == total régional == menages_projetes.csv (84209/101783/120199) -- OK")

somme clusters == total régional == menages_projetes.csv (84209/101783/120199) -- OK


### Source B, tendanciel : monte ou descend entre 2024 et 2050 ?

In [25]:
b_traj = out[out["trajectoire"] == "acces_2050"].groupby("annee")["hh_B"].sum()
b_traj

annee
2024    9325.000000
2035    7720.068247
2050    5481.469265
Name: hh_B, dtype: float64